# ADF (Augmented Dickey-Fuller) Test in Python

A complete walkthrough using `statsmodels` to perform the ADF test: printing the test statistic, p-value, critical values, and automatically making the stationarity decision.

## The `adf_test` Helper Function

This function runs the ADF test on a series and prints a full report, including the hypotheses, decision, and critical values.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller


def adf_test(series, significance=0.05, name="Time Series"):
    """
    Perform Augmented Dickey-Fuller (ADF) test
    and make a statistical decision.
    """

    # Remove missing values
    series = pd.Series(series).dropna()

    # Run ADF test
    result = adfuller(series, autolag="AIC")

    adf_statistic = result[0]
    p_value = result[1]
    used_lags = result[2]
    n_obs = result[3]
    critical_values = result[4]

    print("=" * 60)
    print(f"ADF TEST: {name}")
    print("=" * 60)

    print(f"ADF Statistic : {adf_statistic:.4f}")
    print(f"p-value       : {p_value:.4f}")
    print(f"Used Lags     : {used_lags}")
    print(f"Observations  : {n_obs}")

    print("\nCritical Values:")
    for level, value in critical_values.items():
        print(f"  {level:>4} : {value:.4f}")

    print(f"\nSignificance Level (\u03b1): {significance}")

    # Decision using p-value
    print("\nDecision:")

    if p_value <= significance:
        print("Reject H0")
        print("Conclusion: The series is stationary.")
    else:
        print("Fail to Reject H0")
        print("Conclusion: The series is non-stationary.")

    print("\nHypotheses:")
    print("H0: The series has a unit root (non-stationary).")
    print("H1: The series does not have a unit root (stationary).")

    print("=" * 60)

    return {
        "adf_statistic": adf_statistic,
        "p_value": p_value,
        "critical_values": critical_values,
        "used_lags": used_lags,
        "n_observations": n_obs,
        "stationary": p_value <= significance
    }

## Example 1 — A Stationary Series

Generate white-noise-like data. Since white noise fluctuates around a constant mean with constant variance, we expect the ADF test to **reject H0** (evidence of stationarity).

In [ ]:
np.random.seed(42)

# White-noise-like stationary series
stationary_data = np.random.normal(
    loc=0,
    scale=1,
    size=500
)

result = adf_test(
    stationary_data,
    significance=0.05,
    name="Stationary Series"
)

Expected decision:

```
Reject H0
Conclusion: The series is stationary.
```

## Example 2 — A Non-Stationary Random Walk

Generate a random walk by cumulatively summing normal noise. Random walks are the classic example of a unit-root, non-stationary process, so we expect the test to **fail to reject H0**.

In [ ]:
np.random.seed(42)

# Random walk
random_walk = np.cumsum(
    np.random.normal(
        loc=0,
        scale=1,
        size=500
    )
)

result = adf_test(
    random_walk,
    significance=0.05,
    name="Random Walk"
)

Typical result:

```
Fail to Reject H0
Conclusion: The series is non-stationary.
```

## Applying It to Stock Data

Load a CSV containing a `Close` price column and run the ADF test on the raw closing price.

In [ ]:
df = pd.read_csv("stock_data.csv")

result = adf_test(
    df["Close"],
    significance=0.05,
    name="Stock Closing Price"
)

You might get output like:

```
============================================================
ADF TEST: Stock Closing Price
============================================================
ADF Statistic : -1.8234
p-value       : 0.3692
Used Lags     : 4
Observations  : 495

Critical Values:
   1% : -3.4435
   5% : -2.8673
  10% : -2.5698

Significance Level (α): 0.05

Decision:
Fail to Reject H0
Conclusion: The series is non-stationary.

Hypotheses:
H0: The series has a unit root (non-stationary).
H1: The series does not have a unit root (stationary).
============================================================
```

Raw stock prices are typically non-stationary — this is expected.

## Then Test Returns

Prices are usually non-stationary, but **returns** (percentage changes) frequently are stationary. Let's compute returns and re-run the test.

In [ ]:
df["Return"] = df["Close"].pct_change()

result = adf_test(
    df["Return"],
    significance=0.05,
    name="Stock Returns"
)

You may obtain:

```
ADF Statistic : -18.2345
p-value       : 0.0000

Decision:
Reject H0
Conclusion: The series is stationary.
```

This is one reason financial modeling frequently works with returns rather than raw prices.

## The Workflow

```
Stock Price
     |
     v
  ADF Test
     |
     +-- p > 0.05 --> Fail to reject H0
     |                |
     |                v
     |          Non-stationary
     |                |
     |                v
     |          Difference / Return
     |                |
     |                v
     |             ADF Again
     |
     +-- p <= 0.05 --> Reject H0
                       |
                       v
                    Stationary
```

## Important Interpretation

For the ADF test:

**H0 = Unit root = Non-stationary**

Therefore:

- `p-value <= alpha` &rarr; Reject H0 &rarr; evidence of stationarity
- `p-value > alpha` &rarr; Fail to reject H0 &rarr; insufficient evidence against non-stationarity

> **Note:** For a robust time-series workflow, it's good practice to pair the ADF test with the **KPSS test**, since KPSS has the opposite null hypothesis and provides complementary evidence.